In [4]:
import yfinance as yf
import pandas as pd
import duckdb
from datetime import datetime

In [27]:
tickers = pd.read_excel(r"C:\Users\Incodol\Desktop\proyecto_var\proyecto-de-aula\PORTAFOLIO1.xlsx")


In [16]:
datos = []

for ticker in tickers["ticker"].values:
    msft = yf.Ticker(ticker)
    hist = msft.history(period="1mo")
    hist["activo"]= ticker
    hist["created_at"]= datetime.now()
    hist["updated_at"]= datetime.now()
    hist = hist.reset_index()
    datos.append(hist)


In [17]:
datos_reales = pd.concat(datos)
datos_reales = datos_reales.astype(str)

In [18]:
datos_reales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Date          20 non-null     object
 1   Open          20 non-null     object
 2   High          20 non-null     object
 3   Low           20 non-null     object
 4   Close         20 non-null     object
 5   Volume        20 non-null     object
 6   Dividends     20 non-null     object
 7   Stock Splits  20 non-null     object
 8   activo        20 non-null     object
 9   created_at    20 non-null     object
 10  updated_at    20 non-null     object
dtypes: object(11)
memory usage: 1.8+ KB


In [19]:
con = duckdb.connect("proyecto.duckdb")

In [20]:
con.execute("CREATE SCHEMA IF NOT EXISTS proyecto.bronze")
con.execute("CREATE SCHEMA IF NOT EXISTS proyecto.silver")
con.execute("CREATE SCHEMA IF NOT EXISTS proyecto.gold")

In [23]:
con.execute("""
INSERT INTO proyecto.bronze.precios
    SELECT * FROM datos_reales
""")                   

In [ ]:
con.execute("""
CREATE TABLE IF NOT EXISTS proyecto.silver.precios (
    date TIMESTAMP,
    open DOUBLE,
    high DOUBLE,
    low DOUBLE,
    close DOUBLE,
    volume BIGINT,
    dividends DOUBLE,
    stock_splits DOUBLE,
    activo VARCHAR,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
""")